In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

model_df = pd.read_csv('f1_lap_model_data.csv')


C:\Users\bhavi\AppData\Local\Temp\ipykernel_29676\1412135026.py:7: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  model_df = pd.read_csv('f1_lap_model_data.csv')


In [11]:
import pandas as pd

track_df = pd.read_csv('track_char.csv')
track_df['EventName'] = track_df['EventName'].str.strip()
track_df['TrackDirection'] = track_df['TrackDirection'].str.strip().str.capitalize()

model_events = set(model_df['EventName'].unique())
track_events = set(track_df['EventName'].unique())

print("In model_df but not track_df:", model_events - track_events)
print("In track_df but not model_df:", track_events - model_events)

In model_df but not track_df: set()
In track_df but not model_df: set()


In [12]:

model_df = model_df.merge(track_df, on='EventName', how='left')

# Re-apply clean-data filter (in case it wasn't already applied to this version of model_df)
model_df = model_df.dropna(subset=['LapTime_Seconds', 'TyreLife', 'TrackTemp', 'AirTemp'])

# Then rebuild the split
sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()
print(model_df[track_df.columns].isna().sum())

EventName            0
CircuitLength_km     0
NumCorners           0
NumDRSZones          0
TrackDirection       0
AvgSpeed_kmh         0
ElevationChange_m    0
DownforceLevel       0
dtype: int64


In [13]:
sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()

n_test_races = 5
train_events = event_order[:-n_test_races]
test_events = event_order[-n_test_races:]

train_df = model_df[model_df['EventName'].isin(train_events)]
test_df = model_df[model_df['EventName'].isin(test_events)]

In [14]:
features = ['TyreLife', 'Compound', 'FreshTyre', 'Stint', 'TrackTemp', 
            'AirTemp', 'Driver', 'LapNumber',
            'CircuitLength_km', 'NumCorners', 'NumDRSZones', 
            'TrackDirection', 'AvgSpeed_kmh', 'ElevationChange_m', 'DownforceLevel']

X_train = pd.get_dummies(train_df[features], columns=['Compound', 'Driver', 'TrackDirection', 'DownforceLevel'])
X_test = pd.get_dummies(test_df[features], columns=['Compound', 'Driver', 'TrackDirection', 'DownforceLevel'])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df['LapTime_Seconds']
y_test = test_df['LapTime_Seconds']
print(train_df['LapTime_Seconds'].isna().sum())
print(test_df['LapTime_Seconds'].isna().sum())

0
0


In [17]:

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"MAE: {mae:.3f} seconds")

MAE: 5.691 seconds


In [18]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))

CircuitLength_km         0.685117
AvgSpeed_kmh             0.155121
NumCorners               0.046235
Compound_INTERMEDIATE    0.028824
LapNumber                0.019488
AirTemp                  0.017526
ElevationChange_m        0.014087
TrackTemp                0.008450
TyreLife                 0.005788
Stint                    0.004454
Compound_WET             0.001884
Compound_SOFT            0.001693
NumDRSZones              0.000945
FreshTyre                0.000701
Driver_VER               0.000609
dtype: float64


In [20]:
import joblib
joblib.dump(model, 'lap_time.joblib')
joblib.dump(X_train.columns, 'lap_model_columns.joblib')

['lap_model_columns.joblib']